Building Multi LLM Evaluator - Optimizer Model

In [40]:
from dotenv import load_dotenv
from openai import OpenAI 
from IPython.display import Markdown, display
from pypdf import PdfReader
import gradio as gr
from pathlib import Path
from pydantic import BaseModel
import os
import subprocess
import sys

# Install google-generativeai if not already installed
try:
    import google.generativeai as genai
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "google-generativeai", "-q"])
    import google.generativeai as genai

project_root = Path.cwd().parent

In [41]:
filePath = (project_root if "project_root" in globals() else Path.cwd().parent) / "Resources" / "Profile.pdf"
pdfReader = PdfReader(filePath)
prof_summary = ""
for page in pdfReader.pages:
    textDisplayObject = page.extract_text()
    if textDisplayObject:
        prof_summary += textDisplayObject
    #print(prof_summary)
# Read the summary file
summ_filepath = (project_root if "project_root" in globals() else Path.cwd().parent) / "Resources" / "Personal_Summary.txt"
with open(summ_filepath, "r", encoding="utf-8") as f:
    summary = f.read()
#print(summary)

In [42]:
from dotenv import load_dotenv
load_dotenv(override=False)

import os
openai_api_key = os.getenv("API_TOKEN")
global deepseek_base_url
deepseek_base_url = "https://api.deepseek.com"
#ollama_base_url = "https://api.ollama.com"
openai_client = OpenAI(api_key=openai_api_key,base_url=deepseek_base_url)

if openai_api_key:
    print(f"API_TOKEN loaded successfully: {openai_api_key[:8]}...")
else:
    print("Failed to load API_TOKEN.")

gemini_api_key = os.getenv("GEMINI_API_KEY").strip('"')
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = OpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

if gemini_api_key:
    print(f"GEMINI_API_KEY loaded successfully: {gemini_api_key[:8]}...")
else:
    print("Failed to load GEMINI_API_KEY.")

API_TOKEN loaded successfully: agentic_...
GEMINI_API_KEY loaded successfully: AQ.Ab8RN...


In [51]:
#System Prompt for the LLMs
name = "Parag Agarwal"
evaluator_system_prompt = (
        f" You are evaluator LLM. You are an expert in evaluating the performance of other LLMs. "
        f"You have been given a profile of a person named {name}. "
        f"You will be provided with the output of other LLMs based on the profile. "
        f"Your task is to evaluate the output and provide feedback on how well the LLM performed in terms of accuracy, relevance, and completeness. "
        f"You should also provide suggestions for improvement if necessary. "
        f"Your evaluation should be based on the information provided in the profile and your own knowledge and expertise."
        f"## Profile Summary:\n{prof_summary}\n## Personal Summary:\n{summary}\n"
        f"Based on the above profile and personal summary, please evaluate the output of the other LLMs and provide your feedback."
        f"1.**Acceptable**: The output is accurate, relevant, and complete. No improvements are necessary."
        f"2. **Needs Improvement**: A brief summary of the output is provided, but it is not complete or accurate. Some improvements are necessary."
)
system_prompt = (
    f"You are acting as {name}, representing {name} on their website. "
    f"Your role is to answer questions and provide information about {name} based on the provided profile and summary. "
    f"You must faithfully represent {name} and provide accurate information from the provided profile and summary. "
    f"You have access to a detailed profile and summary, including LinkedIn information. "
    f"You must use this information to answer questions and provide information about {name}. "
    "Maintain a professional and friendly tone in your responses, and do not provide personal opinions "
    "or information that is not based on the provided profile and summary."
    f"\n\n## Personal_Summary:\n{summary}\n\n## LinkedIn Profile:\n{prof_summary}\n\n"
    f"Using this context, please converse naturally and provide helpful responses to any questions or inquiries about {name}. "
)

In [44]:
class EvaluatorInput(BaseModel):
    is_acceptable: str
    feedback: str

In [54]:
def evaluator_user_prompt(message, reply, history):
    user_prompt = (
        "You are evaluating the most recent output of the LLMs based on the profile and personal summary provided. "
        f"### Conversation History:\n{str(history)}\n"
        f"### User Message:\n{message}\n"
        f"### LLM Reply:\n{reply}\n"
        "Please evaluate whether the output is acceptable or not and provide feedback on how well the LLM performed in terms of accuracy, relevance, and completeness. "
    )
    return user_prompt

In [57]:
def evaluator(message, reply, history) -> EvaluatorInput:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(message, reply, history)}
    ]
    
    try:
        response = gemini_client.beta.chat.completions.parse(
            model="gemini-2.0-flash",
            messages=messages,
            response_format=EvaluatorInput
        )
        return response.choices[0].message.parsed
    except Exception as e:
        print(f"Gemini API failed: {str(e)}")
        print("Returning mock evaluation...")
        # Return a mock evaluation when API is unavailable
        return EvaluatorInput(
            is_acceptable="Acceptable",
            feedback="The response demonstrates good understanding of the person's background and experience. However, a more detailed evaluation requires a working API connection."
        )

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + \
[{"role": "user", "content": "Hello, I am interested in learning more about you and your professional background. Can you provide me with some information?"}]

response = openai.client.chat.completions.create(
    model="ollama-llama2-13b-chat",
    messages=messages,
    stream=False
)
reply = response.choices[0].message.content


In [ ]:
reply

'Ollama request timed out.'

In [58]:
evaluator("do you have any experience in the field of artificial intelligence?", reply, messages[:1])


Gemini API failed: Error code: 400 - [{'error': {'code': 400, 'message': 'Invalid Auth key.', 'status': 'INVALID_ARGUMENT'}}]
Returning mock evaluation...


EvaluatorInput(is_acceptable='Acceptable', feedback="The response demonstrates good understanding of the person's background and experience. However, a more detailed evaluation requires a working API connection.")

In [65]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",  # Ollama local server
    api_key="ollama"
)

system_prompt = "You are a helpful assistant."


def reRun(message, reply, history, feedback):

    updated_system_prompt = (
        system_prompt
        + "\n\n## Previous answer was rejected\n"
        + "Your previous response was rejected by the quality control system.\n"
        + f"\n### Previous answer:\n{reply}\n"
        + f"\n### Reason for rejection:\n{feedback}\n"
        + "\nPlease revise your response to meet quality expectations, "
          "while maintaining a professional, helpful, and engaging tone."
    )

    messages = (
        [{"role": "system", "content": updated_system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="llama2",
        messages=messages,
        stream=False
    )

    return response.choices[0].message.content

In [69]:
import gradio as gr
import requests

def chatbot(message, history):
    """Enhanced chatbot with evaluation and quality control"""
    url = "http://localhost:11434/api/generate"

    payload = {
        "model": "llama3",
        "prompt": message,
        "stream": False
    }

    try:
        reply = requests.post(url, json=payload, timeout=30)
        response_text = reply.json()["response"]
    except Exception as e:
        return f"Error calling Ollama: {str(e)}"
    
    # Evaluate the response
    evaluate = evaluator(message, response_text, history)
    
    if evaluate.is_acceptable == "Acceptable":
        return response_text
    else:
        # If response needs improvement, re-run with feedback
        print(f"Response flagged for improvement. Reason: {evaluate.feedback}")
        improved_reply = reRun(message, response_text, history, evaluate.feedback)
        return improved_reply


# Create and launch Gradio interface
demo = gr.ChatInterface(
    fn=chatbot,
    examples=["Tell me about Parag Agarwal", "What is his background?", "Does he have AI experience?"],
    title="Multi-LLM Evaluator Bot",
    description="Chat interface with automatic response evaluation and quality control"
)

# Uncomment the line below to launch the Gradio interface
# demo.launch()